# 01 · Exploratory Data Analysis

Compare the two datasets that make this lab interesting: long, well-formed Amazon reviews vs. short, noisy Sentiment140 tweets.

> Run `make download && make data` first.


In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from nlp_system.data.make_dataset import load_split
pd.set_option('display.max_colwidth', 120)


## Load processed splits


In [ ]:
amazon = load_split('amazon_fine_food', 'train')
tweets = load_split('sentiment140', 'train')
print('Amazon:', amazon.shape, '| Sentiment140:', tweets.shape)
amazon.head(3)


## Class balance


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11,4))
amazon['label'].value_counts().sort_index().plot.bar(ax=ax[0], title='Amazon (0=neg,1=pos)')
tweets['label'].value_counts().sort_index().plot.bar(ax=ax[1], title='Sentiment140')
plt.tight_layout(); plt.show()


## Text length distributions

The defining contrast: review length dwarfs tweet length. This is *why* BM25's length normalization should matter more on Amazon.


In [ ]:
amazon['len'] = amazon['text'].str.split().str.len()
tweets['len'] = tweets['text'].str.split().str.len()
print('Amazon median words:', amazon['len'].median())
print('Tweet  median words:', tweets['len'].median())
fig, ax = plt.subplots(1,2, figsize=(11,4))
amazon['len'].clip(upper=300).hist(bins=50, ax=ax[0]); ax[0].set_title('Amazon word counts')
tweets['len'].clip(upper=60).hist(bins=40, ax=ax[1]); ax[1].set_title('Tweet word counts')
plt.tight_layout(); plt.show()


## Noise inspection

Eyeball the raw noise each preprocessor must handle.


In [ ]:
print('--- Amazon (HTML, run-ons) ---')
for t in amazon['text'].head(3): print(repr(t[:200]))
print('\n--- Tweets (@mentions, urls, #tags, elongation) ---')
for t in tweets['text'].head(5): print(repr(t[:160]))


## Preprocessor effect

Show the configurable pipeline cleaning each domain correctly.


In [ ]:
from nlp_system.pipeline.preprocess import TextPreprocessor
rev = TextPreprocessor.for_reviews(); tw = TextPreprocessor.for_tweets()
print('REVIEW :', amazon['text'].iloc[0][:120])
print('  -> ', rev.transform_one(amazon['text'].iloc[0])[:120])
print('TWEET  :', tweets['text'].iloc[0][:120])
print('  -> ', tw.transform_one(tweets['text'].iloc[0])[:120])
